[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 04](README.md)

# MPI en clúster: Slurm y escalabilidad

**Tema:** 04 · **Sesiones:** 21 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo convertir una ejecución distribuida en un experimento repetible y atribuible a recursos concretos?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** En clúster, el tiempo depende tanto del programa como de la asignación, el mapeo y la red. Un job reproducible registra esos factores.

**Prerrequisitos.**

- Procesos, memoria privada y paso de argumentos.
- Modelo de costo latencia–ancho de banda y referencia serial.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Distinguir asignación Slurm de lanzamiento MPI.
- Calcular escalado fuerte y débil.
- Registrar nodos, tareas, afinidad, versiones y repeticiones.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

La reserva define recursos; `srun` o `mpiexec` inicia procesos según la integración del clúster.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

El mapeo proceso–núcleo–NUMA debe registrarse porque cambia comunicación y memoria.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Los resultados válidos incluyen script, job id, módulos, entrada, tiempos crudos y manifiesto.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- rank — identidad de un proceso dentro de un comunicador
- tag — etiqueta que participa en el emparejamiento de mensajes
- colectiva — operación coordinada por todos los procesos del comunicador


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Mpi Comunicacion

![Procesos MPI con mensajes y colectiva](../../images/mpi-comunicacion.svg)

**Cómo leerlo.** Las flechas exteriores representan punto a punto; las interiores, coordinación colectiva. Todos los ranks deben respetar comunicador, orden y contrato de datos.

### Escalabilidad

![Curvas ideal, limitada y observada](../../images/escalabilidad.svg)

**Cómo leerlo.** Compara la pendiente de cada curva y pregunta siempre si el tamaño es fijo. La distancia frente a la línea ideal se interpreta con eficiencia y overhead.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "04"
NOTEBOOK = "04_mpi/03_escalabilidad_slurm.ipynb"
assert (ROOT / "curso" / "notebooks" / "04_mpi" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Escalado fuerte

**Situación.** Se calculan speedup y eficiencia de una serie sintética.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
times = {1: 84.0, 2: 44.5, 4: 24.0, 8: 14.2, 16: 10.1}
baseline = times[1]
assert list(times) == [1, 2, 4, 8, 16]
for p, elapsed in times.items():
    speedup = baseline / elapsed
    efficiency = speedup / p
    assert 0 < efficiency <= 1
    print(f"p={p:2} tiempo={elapsed:5.1f}s speedup={speedup:5.2f} eficiencia={efficiency:5.3f}")


### Explicación del resultado

La pérdida de eficiencia debe relacionarse con comunicación, desbalance o saturación mediante mediciones adicionales.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Plan de trabajos

**Situación.** Se construye una matriz de recursos sin ejecutar el planificador.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
plan = []
for nodes in (1, 2, 4):
    for tasks_per_node in (1, 2, 4):
        plan.append({"nodes": nodes, "ntasks_per_node": tasks_per_node, "total_ranks": nodes*tasks_per_node, "repetitions": 5})
assert len(plan) == 9
for row in plan: print(row)


### Lectura razonada

Cada punto debe ejecutarse con la misma entrada, política de afinidad y versión del binario.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué diferencia hay entre reservar recursos con Slurm y lanzar procesos MPI sobre esa reserva?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Preparar un script `sbatch` con salida que incluya job id y hostnames.
2. Ejecutar repeticiones evitando mezclar calentamiento.
3. Conservar CSV/JSON y manifiesto junto con la gráfica.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Usar nodos asignados de manera interactiva sin registrar opciones.
- Comparar jobs con frecuencias o afinidades distintas.
- Presentar solo speedup sin tiempos crudos.


## Criterios de aceptación

- Script y configuración Slurm versionados.
- Cinco o más repeticiones por punto.
- Escalado acompañado de dispersión y explicación.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo convertir una ejecución distribuida en un experimento repetible y atribuible a recursos concretos?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Scripts MPI](../../../mpi/)
- [Protocolo de evidencia](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md#8-niveles-de-evidencia)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 04](README.md)
